In [1]:
import sys
import os
import glob

import re

import itertools

import numpy as np
import pandas as pd

from tqdm.notebook import tqdm

import torch
from torch.utils.data import DataLoader
import pytorch_lightning as pl

In [2]:
sys.path.append("../model/")
import utrdata_cl as utrdata
from legnet import LegNetClassifier
from pl_regressor import RNARegressor

## Loading data

In [3]:
utr_type = "utr3"

In [4]:
if utr_type == "utr5":
    MODEL_PATH = "../regression_multiple/saved_models/model-utr5-deltas-epoch=9-step=840.ckpt"
elif utr_type == "utr3":
    MODEL_PATH = "../regression_multiple/saved_models/model-utr3-deltas-epoch=9-step=1330.ckpt"
else:
    raise ValueError

In [5]:
PATH_FROM = f"./{utr_type.upper()}_zscores_2024-06-04.csv"
df = pd.read_csv(PATH_FROM)

In [6]:
assert (df["seq"] == df["seq"].str.upper()).all()

In [7]:
ct_classes = df["cell_type"].unique()
num_classes = ct_classes.shape[0]
num_classes

5

In [8]:
batch_size = 1024
num_workers = 32

In [9]:
test_set = utrdata.UTRData(
    df=df,
    predict_cols=[],
    features=("sequence", "positional", "conditions"),
    construct_type=utr_type,
    augment=False,
    augment_test_time=False,
    augment_kws=dict(
        extend_left=0,
        extend_right=0,
        shift_left=0,
        shift_right=0,
        revcomp=False,
    ),
)
dl_test = DataLoader(
    test_set,
    batch_size=batch_size,
    num_workers=num_workers,
    shuffle=False,
    drop_last=False
)

model = RNARegressor.load_from_checkpoint(MODEL_PATH)
trainer = pl.Trainer(
    callbacks=[pl.callbacks.TQDMProgressBar(refresh_rate=0.5)],
    logger=False,
    accelerator="gpu",
    devices=[0],
    deterministic=True,
    num_sanity_val_steps=0,
)
prediction = trainer.predict(model=model, dataloaders=dl_test)
test_pred, test_real = zip(*prediction)
test_pred = torch.concat(test_pred).numpy()
test_real = torch.concat(test_real).numpy()
df_pred = df.copy()
df_pred["pred_delta"] = test_pred[:, 0]
df_pred["pred_mass_center"] = test_pred[:, 1]

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting: 0it [00:00, ?it/s]

In [10]:
df_pred

,seq,cell_type,1,2,3,4,mass_center,mass_center_mean,diff,zscore,mass_center_std,pred_delta,pred_mass_center
0,AAAAAAAAAAAAAAAAAAAAAAAAACACAAATACATTCACCCACCT...,c1,7803.0,7567.0,6317.0,6285.0,2.396253,2.381483,0.014770,0.153872,0.095990,0.201341,2.277320
1,AAAAAAAAAAAAAAAAAAAAAAAAACACAAATACATTCACCCACCT...,c17,11655.0,10389.0,7803.0,6142.0,2.234294,2.381483,-0.147189,-1.533391,0.095990,-0.035654,2.070518
2,AAAAAAAAAAAAAAAAAAAAAAAAACACAAATACATTCACCCACCT...,c2,6274.0,5816.0,6332.0,6061.0,2.497488,2.381483,0.116005,1.208515,0.095990,0.006147,2.095751
3,AAAAAAAAAAAAAAAAAAAAAAAAACACAAATACATTCACCCACCT...,c4,8586.0,7638.0,8481.0,5550.0,2.363411,2.381483,-0.018072,-0.188273,0.095990,0.000262,2.057197
4,AAAAAAAAAAAAAAAAAAAAAAAAACACAAATACATTCACCCACCT...,c6,2888.0,2576.0,2669.0,2274.0,2.415970,2.381483,0.034487,0.359276,0.095990,-0.011893,2.293071
...,...,...,...,...,...,...,...,...,...,...,...,...,...
28370,TTTTTTTTTTTTTTTTTTTTTTTTCCTTCCAAACTCTCCACAAACT...,c1,1759.0,650.0,264.0,1164.0,2.217097,2.192395,0.024702,0.057992,0.425958,-0.138581,2.354964
28371,TTTTTTTTTTTTTTTTTTTTTTTTCCTTCCAAACTCTCCACAAACT...,c17,1391.0,2933.0,4367.0,3348.0,2.803389,2.192395,0.610994,1.434402,0.425958,0.045548,2.522268
28372,TTTTTTTTTTTTTTTTTTTTTTTTCCTTCCAAACTCTCCACAAACT...,c2,150.0,74.0,29.0,9.0,1.606870,2.192395,-0.585524,-1.374607,0.425958,-0.052428,2.379684
28373,TTTTTTTTTTTTTTTTTTTTTTTTCCTTCCAAACTCTCCACAAACT...,c4,6677.0,4461.0,3281.0,2592.0,2.105108,2.192395,-0.087286,-0.204917,0.425958,0.060710,2.535279


## Adding metadata

In [11]:
df_src = pd.read_table(f"../../data/lib2/raw/{utr_type.upper()}_sequence_metadata_06_04_24.tsv", index_col=0)
df.index.name = "seq"

In [12]:
def get_tag_rowwise(row: pd.Series):
    ct1 = row['first_cell_line']
    ct2 = row['second_cell_line']
    if pd.isna(ct1):
        return {"direction": np.nan, "cell_types": np.nan}
    elif pd.isna(ct2):
        if ct1 in {"up", "down"}:
            return {"direction": ct1, "cell_types": np.nan}
        else:
            raise Exception
    else:
        return {"direction": np.nan, "cell_types": f"{ct1}-{ct2}"}

In [13]:
tags_df = df_src.apply(get_tag_rowwise, axis=1).apply(pd.Series)

### Info for diffusion

In [14]:
diffusion_files = glob.glob(f"../../generator/diffusion/generate/results/*{utr_type[-1]}UTR.csv")
diffusion_taken = []
for gen_result_file in tqdm(diffusion_files):
    bn = os.path.basename(gen_result_file)
    ct = re.match(r"cell_(c\d+)_epoch_\d+_\dUTR.csv", bn).group(1)
    gen_result_df = pd.read_csv(gen_result_file)
    gen_result_df = gen_result_df[gen_result_df["seq"].isin(set(df["seq"]))]
    gen_result_df["diffusion_cell_type"] = ct
    diffusion_taken.append(gen_result_df)
diffusion_taken = pd.concat(diffusion_taken).set_index("seq")

  0%|          | 0/5 [00:00<?, ?it/s]

In [15]:
assert not diffusion_taken.index.duplicated().any()

## Reshaping data

In [16]:
df_sub = df_pred.set_index(["seq", "cell_type"]).copy()
df_new = df_sub[["mass_center", "diff", "pred_mass_center", "pred_delta"]]
df_new = df_new.unstack(1).reorder_levels((1, 0), axis=1).sort_index(axis=1)

In [17]:
df_new["source"] = df_src["source"]
df_new["source_ext"] = df_src["name"]
df_new["diffusion_query"] = diffusion_taken["mass_center"]
df_new["diffusion_cell_type"] = diffusion_taken["diffusion_cell_type"]
df_new["direction"] = tags_df["direction"]
df_new["cell_types_directional"] = tags_df["cell_types"]
df_new["cell_types_ordered"] = np.nan

In [18]:
cell_types = ["c1", "c2", "c4", "c6", "c13", "c17"]
for ct1, ct2 in itertools.combinations(cell_types, 2):
    df_new.loc[df_new["cell_types_directional"] == f"{ct1}-{ct2}", "cell_types_ordered"] = f"{ct1}-{ct2}"
    df_new.loc[df_new["cell_types_directional"] == f"{ct1}-{ct2}", "direction"] = "up"
    df_new.loc[df_new["cell_types_directional"] == f"{ct2}-{ct1}", "cell_types_ordered"] = f"{ct1}-{ct2}"
    df_new.loc[df_new["cell_types_directional"] == f"{ct2}-{ct1}", "direction"] = "down"

In [19]:
df_new

cell_type                                                 c1              \
                                                        diff mass_center   
seq                                                                        
AAAAAAAAAAAAAAAAAAAAAAAAACACAAATACATTCACCCACCTC...  0.014770    2.396253   
AAAAAAAAAAAAAAAAAAAAAAAACAAATCCCTGTGCCTCCTCCCTC...  0.008364    2.406223   
AAAAAAAAAAAAAAAAAAAAAAAATGTGTTCTCATCTCCCTTTCTCC... -0.066301    2.121921   
AAAAAAAAAAAAAAAAAAAAAAATACATCCACCCCCACCCCCCCCCC...  0.115393    2.574347   
AAAAAACCTCCTAGGTTAAAATAGTTTCTTTATTTAAGATAGAATAA...  0.445065    2.621438   
...                                                      ...         ...   
TTTTTCTAGGCCTAGTGCTACGCTTAAAGTAAATGGTAAATTAAGTA... -0.435868    2.199873   
TTTTTGCGGGGCAAAGCTGTTCATTGTAACAAAATTCAATCAAAAAG... -0.432974    2.560125   
TTTTTTCCCTCTTTCCTTTTCCCCCTTTTTCTCTTTCTTTCCTTTCT...  0.434538    3.610602   
TTTTTTTTTTTTAAATTTAAAATATTTTTTTGATTGGAGCAAAACAG...  0.577643    2.837802   
TTTTTTTTTTTTTTTTTTTTTTTTCCTTCCAAACTCTCCACAAACTC...  0.024702    2.217097   

cell_type                                                      \
                                                   pred_delta   
seq                                                             
AAAAAAAAAAAAAAAAAAAAAAAAACACAAATACATTCACCCACCTC...   0.201341   
AAAAAAAAAAAAAAAAAAAAAAAACAAATCCCTGTGCCTCCTCCCTC...   0.175673   
AAAAAAAAAAAAAAAAAAAAAAAATGTGTTCTCATCTCCCTTTCTCC...   0.025114   
AAAAAAAAAAAAAAAAAAAAAAATACATCCACCCCCACCCCCCCCCC...   0.216479   
AAAAAACCTCCTAGGTTAAAATAGTTTCTTTATTTAAGATAGAATAA...  -0.041919   
...                                                       ...   
TTTTTCTAGGCCTAGTGCTACGCTTAAAGTAAATGGTAAATTAAGTA...   0.110044   
TTTTTGCGGGGCAAAGCTGTTCATTGTAACAAAATTCAATCAAAAAG...  -0.013880   
TTTTTTCCCTCTTTCCTTTTCCCCCTTTTTCTCTTTCTTTCCTTTCT...   0.173421   
TTTTTTTTTTTTAAATTTAAAATATTTTTTTGATTGGAGCAAAACAG...   0.032756   
TTTTTTTTTTTTTTTTTTTTTTTTCCTTCCAAACTCTCCACAAACTC...  -0.138581   

cell_type                                                                 c17  \
                                                   pred_mass_center      diff   
seq                                                                             
AAAAAAAAAAAAAAAAAAAAAAAAACACAAATACATTCACCCACCTC...         2.277320 -0.147189   
AAAAAAAAAAAAAAAAAAAAAAAACAAATCCCTGTGCCTCCTCCCTC...         2.371295 -0.170683   
AAAAAAAAAAAAAAAAAAAAAAAATGTGTTCTCATCTCCCTTTCTCC...         2.560409 -0.214635   
AAAAAAAAAAAAAAAAAAAAAAATACATCCACCCCCACCCCCCCCCC...         2.342797  0.026425   
AAAAAACCTCCTAGGTTAAAATAGTTTCTTTATTTAAGATAGAATAA...         2.438685 -0.334041   
...                                                             ...       ...   
TTTTTCTAGGCCTAGTGCTACGCTTAAAGTAAATGGTAAATTAAGTA...         2.441337  0.711035   
TTTTTGCGGGGCAAAGCTGTTCATTGTAACAAAATTCAATCAAAAAG...         2.731053  0.582082   
TTTTTTCCCTCTTTCCTTTTCCCCCTTTTTCTCTTTCTTTCCTTTCT...         2.922889 -0.094429   
TTTTTTTTTTTTAAATTTAAAATATTTTTTTGATTGGAGCAAAACAG...         2.578005 -0.018955   
TTTTTTTTTTTTTTTTTTTTTTTTCCTTCCAAACTCTCCACAAACTC...         2.354964  0.610994   

cell_type                                                                  \
                                                   mass_center pred_delta   
seq                                                                         
AAAAAAAAAAAAAAAAAAAAAAAAACACAAATACATTCACCCACCTC...    2.234294  -0.035654   
AAAAAAAAAAAAAAAAAAAAAAAACAAATCCCTGTGCCTCCTCCCTC...    2.227175  -0.016663   
AAAAAAAAAAAAAAAAAAAAAAAATGTGTTCTCATCTCCCTTTCTCC...    1.973587  -0.159187   
AAAAAAAAAAAAAAAAAAAAAAATACATCCACCCCCACCCCCCCCCC...    2.485379  -0.082748   
AAAAAACCTCCTAGGTTAAAATAGTTTCTTTATTTAAGATAGAATAA...    1.842332  -0.006382   
...                                                        ...        ...   
TTTTTCTAGGCCTAGTGCTACGCTTAAAGTAAATGGTAAATTAAGTA...    3.346776   0.151544   
TTTTTGCGGGGCAAAGCTGTTCATTGTAACAAAATTCAATCAAAAAG...    3.575182   0.105321   
TTTTTTCCCTCTTTCCTTTTCCCCCTTTTTCTCTTTCTTTCC

In [20]:
df_new.to_parquet(f"{utr_type.upper()}_library2_preds.parquet")

---